In [0]:
from pyspark.sql import SparkSession, functions as F
import logging

# Set up configuration and schema path
catalog_name = "workspace" 
schema_name = "capstone_project"
full_path = f"{catalog_name}.{schema_name}"

# Initialize logging for the gold layer process
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("GoldLayer")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {full_path}")

class GoldMetricAggregator:
    def __init__(self, spark, schema_path):
        self.spark = spark
        self.schema_path = schema_path

    def fetch_silver_table(self, table_name):
        """Load a table from the silver layer."""
        return self.spark.read.table(f"{self.schema_path}.silver_{table_name}_final")

    def create_pivoted_comparison(self, df, metric_name, filter_col=None, filter_val=None):
        """Create a pivoted table with columns for sector and year combinations."""
        # Determine the area column name dynamically
        area_col = "Total__Rural__Urban" if "Total__Rural__Urban" in df.columns else "Total_Rural_Urban"
        
        # Apply filter if specific status is required (e.g., Literate or Employed)
        if filter_col and filter_val:
            df = df.filter(F.lower(F.col(filter_col)) == filter_val.lower())
        
        # Create a helper column for pivoting (e.g., Rural_2011)
        pivot_df = df.withColumn("Pivot_Key", F.concat(F.col(area_col), F.lit("_"), F.col("Year")))
        
        # Group by State and pivot based on the sector and year
        comparison_table = pivot_df.groupBy("State_Name").pivot("Pivot_Key").agg(F.sum("Value"))
        
        # Save the result as a gold table
        table_name = f"gold_pivoted_{metric_name}_comparison"
        comparison_table.write.format("delta").mode("overwrite").saveAsTable(f"{self.schema_path}.{table_name}")
        
        print(f"Success: Created {table_name}")
        return comparison_table

# Execution phase for creating the comparative aggregates
try:
    gold_engine = GoldMetricAggregator(spark, full_path)
    
    # Process population comparison
    pop_df = gold_engine.fetch_silver_table("population")
    gold_engine.create_pivoted_comparison(pop_df, "population")
    
    # Process literacy comparison for literate population only
    lit_df = gold_engine.fetch_silver_table("literacy")
    gold_engine.create_pivoted_comparison(lit_df, "literacy", "Literacy_Status", "Literate")
    
    # Process employment comparison for employed population only
    emp_df = gold_engine.fetch_silver_table("employment")
    gold_engine.create_pivoted_comparison(emp_df, "employment", "Employment_Status", "Employed")

    print("\nGold layer processing completed for all pivoted comparison tables.")

except Exception as err:
    logger.error(f"Gold layer transformation failure: {err}")

# Preview block to validate the data structure
print("\nPreview of population comparison records")
spark.read.table(f"{full_path}.gold_pivoted_population_comparison").show(5)

print("\nPreview of literacy comparison records")
spark.read.table(f"{full_path}.gold_pivoted_literacy_comparison").show(5)

print("\nPreview of employment comparison records")
spark.read.table(f"{full_path}.gold_pivoted_employment_comparison").show(5)


Success: Created gold_pivoted_population_comparison
Success: Created gold_pivoted_literacy_comparison
Success: Created gold_pivoted_employment_comparison

Gold layer processing completed for all pivoted comparison tables.

Preview of population comparison records
+--------------------+----------+----------+----------+----------+----------+----------+
|          State_Name|Rural_2011|Rural_2021|Total_2011|Total_2021|Urban_2011|Urban_2021|
+--------------------+----------+----------+----------+----------+----------+----------+
|Andaman and Nicob...|   6617791|   3351006|  11069605|   5929394|   4451814|   2578388|
|      Andhra Pradesh|  28301237|  23265889|  46884551|  35651375|  18583314|  12385486|
|               Delhi|  19158362|  21363443|  30388939|  34435512|  11230577|  13072069|
|         Maharashtra|  73800436|  74234078| 113990319| 120831927|  40189883|  46597849|
|               Bihar|  75320778|  68862022| 121978245| 110729106|  46657467|  41867084|
+--------------------+--

In [0]:
try:
    # Fetch the silver population table
    pop_df = spark.read.table(f"{full_path}.silver_population_final")
    
    # Determine the area column name dynamically
    area_col = "Total__Rural__Urban" if "Total__Rural__Urban" in pop_df.columns else "Total_Rural_Urban"

    print("Metric: Generating ethnic division comparison by sector and year")

    # Create a specialized pivot key: EthnicGroup_Sector_Year (e.g., Sc_Rural_2011)
    # We use initcap to ensure consistent naming in the column headers
    ethnic_pivot_df = pop_df.withColumn(
        "Pivot_Key", 
        F.concat(
            F.initcap(F.col("Ethnic_Group")), F.lit("_"), 
            F.initcap(F.col(area_col)), F.lit("_"), 
            F.col("Year")
        )
    )

    # Group by State and pivot based on the ethnic breakdown
    ethnic_comparison = ethnic_pivot_df.groupBy("State_Name") \
        .pivot("Pivot_Key") \
        .agg(F.sum("Value")) \
        .na.fill(0)

    # Save as a specific gold table
    target_table = f"{full_path}.gold_pivoted_ethnic_division_comparison"
    ethnic_comparison.write.format("delta").mode("overwrite").saveAsTable(target_table)

    print(f"Success: Created {target_table}")

    # Preview the resulting structure
    print("\nPreview of ethnic division comparison records")
    spark.read.table(target_table).show(5, truncate=False)

except Exception as err:
    print(f"Error creating ethnic division table: {err}")

Metric: Generating ethnic division comparison by sector and year
Success: Created workspace.capstone_project.gold_pivoted_ethnic_division_comparison

Preview of ethnic division comparison records
+---------------------------+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+---------------+---------------+---------------+---------------+---------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+----------------+----------------+----------------+----------------+----------------+----------------+---------------+---------------+---------------+---------------+---------------+---------------+
|State_Name             

In [0]:
class DistrictMetricGenerator:
    def __init__(self, spark, schema_path):
        self.spark = spark
        self.schema_path = schema_path

    def fetch_silver(self, table_name):
        """Fetch the final silver table."""
        return self.spark.read.table(f"{self.schema_path}.silver_{table_name}_final")

    def create_district_pivot(self, df, metric_name, filter_col=None, filter_val=None, pivot_on_ethnic=False):
        """Generic function to create district-level pivoted tables."""
        area_col = "Total__Rural__Urban" if "Total__Rural__Urban" in df.columns else "Total_Rural_Urban"
        
        # Apply filters for specific metrics like Literate or Employed
        if filter_col and filter_val:
            df = df.filter(F.lower(F.col(filter_col)) == filter_val.lower())
        
        # Create pivot key based on Sector, Year, and optionally Ethnic Group
        if pivot_on_ethnic:
            df = df.withColumn("Pivot_Key", F.concat(F.col("Ethnic_Group"), F.lit("_"), F.col(area_col), F.lit("_"), F.col("Year")))
        else:
            df = df.withColumn("Pivot_Key", F.concat(F.col(area_col), F.lit("_"), F.col("Year")))
        
        # Group by State and District to ensure unique identification
        district_pivot = df.groupBy("State_Name", "District_Name") \
            .pivot("Pivot_Key") \
            .agg(F.sum("Value")) \
            .na.fill(0)
        
        # Save table with 'district' prefix
        target_name = f"gold_district_pivoted_{metric_name}_comparison"
        district_pivot.write.format("delta").mode("overwrite").saveAsTable(f"{self.schema_path}.{target_name}")
        
        print(f"Success: Created {target_name}")
        return district_pivot

# Execution phase for district-level metrics
try:
    generator = DistrictMetricGenerator(spark, full_path)
    
    # 1. District Population Comparison
    pop_df = generator.fetch_silver("population")
    generator.create_district_pivot(pop_df, "population")
    
    # 2. District Literacy Comparison
    lit_df = generator.fetch_silver("literacy")
    generator.create_district_pivot(lit_df, "literacy", "Literacy_Status", "Literate")
    
    # 3. District Employment Comparison
    emp_df = generator.fetch_silver("employment")
    generator.create_district_pivot(emp_df, "employment", "Employment_Status", "Employed")
    
    # 4. District Ethnic Division Comparison
    generator.create_district_pivot(pop_df, "ethnic_division", pivot_on_ethnic=True)

    print("\nDistrict-wise gold layer metrics generated successfully.")

except Exception as err:
    print(f"Error creating district metrics: {err}")

# Preview block for verification
print("\nPreview of district-level population comparison")
spark.read.table(f"{full_path}.gold_district_pivoted_population_comparison").show(5)

Success: Created gold_district_pivoted_population_comparison
Success: Created gold_district_pivoted_literacy_comparison
Success: Created gold_district_pivoted_employment_comparison
Success: Created gold_district_pivoted_ethnic_division_comparison

District-wise gold layer metrics generated successfully.

Preview of district-level population comparison
+-----------------+-------------+----------+----------+----------+----------+----------+----------+
|       State_Name|District_Name|Rural_2011|Rural_2021|Total_2011|Total_2021|Urban_2011|Urban_2021|
+-----------------+-------------+----------+----------+----------+----------+----------+----------+
|Arunachal Pradesh|  Upper Siang|   1498883|   2825945|   2172851|   4280547|    673968|   1454602|
|   Andhra Pradesh|    Anantapur|   3572862|   1608640|   5267831|   2419425|   1694969|    810785|
|   Andhra Pradesh|East Godavari|   1345378|    670760|   2316634|    977251|    971256|    306491|
|            Bihar|  Muzaffarpur|   2551791|  

In [0]:
print("District-level gold layer validation: Literacy, Employment, and Ethnic Division")

# Preview the literacy comparison table at the district level
print("\nPreview of literacy comparison records")
spark.read.table(f"{full_path}.gold_district_pivoted_literacy_comparison").show(5, truncate=False)

# Preview the employment comparison table at the district level
print("\nPreview of employment comparison records")
spark.read.table(f"{full_path}.gold_district_pivoted_employment_comparison").show(5, truncate=False)

# Preview the ethnic division comparison table at the district level
print("\nPreview of ethnic division comparison records")
spark.read.table(f"{full_path}.gold_district_pivoted_ethnic_division_comparison").show(5, truncate=False)

District-level gold layer validation: Literacy, Employment, and Ethnic Division

Preview of literacy comparison records
+-----------+----------------+----------+----------+----------+----------+----------+----------+
|State_Name |District_Name   |Rural_2011|Rural_2021|Total_2011|Total_2021|Urban_2011|Urban_2021|
+-----------+----------------+----------+----------+----------+----------+----------+----------+
|Assam      |Kamrup          |2357599   |2419015   |3546202   |3920229   |1188603   |1501214   |
|Delhi      |North East Delhi|1591839   |2323387   |2309637   |4431393   |717798    |2108006   |
|Bihar      |Kishanganj      |914226    |311431    |1507101   |607438    |592875    |296007    |
|Maharashtra|Sindhudurg      |2616621   |1212711   |4227156   |1859761   |1610535   |647050    |
|Bihar      |Siwan           |930544    |563970    |1435460   |946556    |504916    |382586    |
+-----------+----------------+----------+----------+----------+----------+----------+----------+
only sh